In [18]:
import pandas as pd
from pathlib import Path

# After the May 2026 reorg, supplementary files live in subfolders:
#   Iteration_1/Step3_Manual_Labeling/  ← raw consensus from Step 3
#   Iteration_1/Step5_Analysis/         ← translated consensus + analysis artifacts
folder = Path.cwd()
translated_path = folder / 'Iteration_1' / 'Step5_Analysis' / 'iteration_1_consensus_translated.csv'
consensus_path  = folder / 'Iteration_1' / 'Step3_Manual_Labeling' / 'iteration_1_labels_consensus.csv'

if translated_path.exists():
    df = pd.read_csv(translated_path)
    print(f"Loaded {len(df)} users from translated file")
else:
    df = pd.read_csv(consensus_path)
    print(f"Loaded {len(df)} users from raw consensus (run cell 1 to translate)")

print(f"Columns: {list(df.columns)}")

Loaded 100 users from translated file
Columns: ['username', 'profile_url', 'display_name', 'description', 'location', 'followers_count', 'following_count', 'statuses_count', 'created_at', 'target_population', 'locals_vs_diaspora', 'person_vs_organization', 'consensus_source', 'description_en', 'display_name_en']


In [ ]:
# Translate description + display_name to English (skips if already done).
if {'description_en', 'display_name_en'}.issubset(df.columns):
    print("Translations already present — skipping.")
else:
    from deep_translator import GoogleTranslator
    import time

    def translate_safe(text, target='en'):
        """Translate text to English; return original on failure or empty input."""
        if pd.isna(text) or str(text).strip() == '':
            return ''
        try:
            return GoogleTranslator(source='auto', target=target).translate(str(text)[:4500])
        except Exception:
            return str(text)

    print("Translating descriptions...")
    df['description_en'] = ''
    for i, val in enumerate(df['description']):
        df.at[i, 'description_en'] = translate_safe(val)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(df)} done")
        time.sleep(0.1)

    print("\nTranslating display names...")
    df['display_name_en'] = ''
    for i, val in enumerate(df['display_name']):
        df.at[i, 'display_name_en'] = translate_safe(val)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(df)} done")
        time.sleep(0.1)

    translated_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(translated_path, index=False)
    print(f"\nSaved: {translated_path.name}")

df[['username', 'description', 'description_en', 'display_name', 'display_name_en']].head(5)

In [12]:
# One-time installs. Run only if a module is missing.
# %pip install deep-translator scikit-learn xgboost scipy
# On macOS, XGBoost also needs OpenMP — run this in a terminal:  brew install libomp

In [13]:
import numpy as np

# ---- Numeric features ----
# Fill NaN counts with 0
for c in ['followers_count', 'following_count', 'statuses_count']:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

# Bio length (chars) — use translated bio
df['bio_length'] = df['description_en'].fillna('').astype(str).str.len()

# Followers / following ratio (+1 to avoid divide by zero)
df['followers_following_ratio'] = df['followers_count'] / (df['following_count'] + 1)

# Account age in years
df['created_at_dt'] = pd.to_datetime(df['created_at'], format='%B %Y', errors='coerce')
today = pd.Timestamp('2026-05-14')
df['account_age_years'] = ((today - df['created_at_dt']).dt.days / 365.25).fillna(0)

# Has description / has location (binary)
df['has_description'] = df['description'].notna().astype(int)
df['has_location']    = df['location'].notna().astype(int)

# Domain-specific: does bio/name/location/username mention Iran?
iran_keywords = ['iran', 'iranian', 'persian', 'persia', 'tehran', 'shiraz', 'esfahan',
                 'isfahan', 'mashhad', 'tabriz', 'kerman', 'qom', 'farsi']
def contains_iran(text):
    if pd.isna(text):
        return 0
    return int(any(kw in str(text).lower() for kw in iran_keywords))

df['bio_mentions_iran']      = df['description_en'].apply(contains_iran)
df['name_mentions_iran']     = df['display_name_en'].apply(contains_iran)
df['location_mentions_iran'] = df['location'].apply(contains_iran)

numeric_features = [
    'followers_count', 'following_count', 'statuses_count',
    'followers_following_ratio', 'bio_length', 'account_age_years',
    'has_description', 'has_location',
    'bio_mentions_iran', 'name_mentions_iran', 'location_mentions_iran',
]

print(f"Numeric features built: {len(numeric_features)}")
print(df[numeric_features].describe().round(2))

Numeric features built: 11
       followers_count  following_count  statuses_count  \
count           100.00           100.00          100.00   
mean           1006.31           921.75         1383.85   
std            2014.59          1456.92         2038.59   
min               0.00             0.00            0.00   
25%               0.00            61.50            7.00   
50%              26.50           349.50          411.50   
75%             543.25           812.00         1401.50   
max            8556.00          7496.00         8467.00   

       followers_following_ratio  bio_length  account_age_years  \
count                     100.00      100.00             100.00   
mean                       55.95       65.58               7.55   
std                       350.02       61.87               5.81   
min                         0.00        0.00               0.45   
25%                         0.00        0.00               2.10   
50%                         0.09       

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

def build_tfidf(df, columns, max_features=300, min_df=2):
    """Concatenate the given columns, then vectorize with TF-IDF (one fresh vectorizer per call)."""
    combined = df[columns[0]].fillna('').astype(str)
    for c in columns[1:]:
        combined = combined + ' ' + df[c].fillna('').astype(str)

    vectorizer = TfidfVectorizer(
        max_features=max_features,
        lowercase=True,
        stop_words='english',
        ngram_range=(1, 1),
        min_df=min_df,
    )
    matrix = vectorizer.fit_transform(combined)
    return matrix, vectorizer

# Build the 7 TF-IDF variants the spec lists (PDF page 12)
# min_df=1 for username/fullname only — they have too little shared vocab
tfidf_sets = {
    'desc':                build_tfidf(df, ['description_en'],                            min_df=2),
    'username':            build_tfidf(df, ['username'],                                  min_df=1),
    'fullname':            build_tfidf(df, ['display_name_en'],                           min_df=1),
    'desc_user':           build_tfidf(df, ['description_en', 'username'],                min_df=2),
    'desc_fullname':       build_tfidf(df, ['description_en', 'display_name_en'],         min_df=2),
    'user_fullname':       build_tfidf(df, ['username', 'display_name_en'],               min_df=1),
    'desc_user_fullname':  build_tfidf(df, ['description_en', 'username', 'display_name_en'], min_df=2),
}

print("TF-IDF feature sets:")
for name, (matrix, vec) in tfidf_sets.items():
    print(f"  {name:<22s}  shape={matrix.shape}  vocab_size={len(vec.vocabulary_)}")

numeric_df = df[numeric_features].copy()
print(f"\nNumeric features matrix: {numeric_df.shape}")

print("\nLabel counts:")
print(f"  target_population:      {dict(df['target_population'].value_counts().sort_index())}")
print(f"  locals_vs_diaspora:     {dict(df['locals_vs_diaspora'].value_counts().sort_index())}")
print(f"  person_vs_organization: {dict(df['person_vs_organization'].value_counts().sort_index())}")

TF-IDF feature sets:
  desc                    shape=(100, 51)  vocab_size=51
  username                shape=(100, 100)  vocab_size=100
  fullname                shape=(100, 173)  vocab_size=173
  desc_user               shape=(100, 51)  vocab_size=51
  desc_fullname           shape=(100, 64)  vocab_size=64
  user_fullname           shape=(100, 268)  vocab_size=268
  desc_user_fullname      shape=(100, 64)  vocab_size=64

Numeric features matrix: (100, 11)

Label counts:
  target_population:      {0: np.int64(50), 1: np.int64(13), 2: np.int64(37)}
  locals_vs_diaspora:     {0: np.int64(2), 1: np.int64(6), 2: np.int64(92)}
  person_vs_organization: {0: np.int64(19), 1: np.int64(48), 2: np.int64(33)}


## Step 5 — Full experiment grid

PDF page 12–13: train at least 6 algorithms (LogReg, DecisionTree, RandomForest, SVM, XGBoost, AdaBoost) on multiple feature sets, with **two validation strategies** (K=5 K-Fold and LOOCV), in **two balance modes** (balanced / unbalanced), on **both 3-class and 2-class** versions of each target.

Per row in `experiments_results_iteration_1.csv` (page 13 column spec):
`iteration, target_column, #classes, #class_0..2, min_class_size, training_type, K, algorithm, feature_set, Features_count, balanced, accuracy, precision, recall, F1, AUC`

In [15]:
# --- Imports for the experiment loop ---
import numpy as np
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, LeaveOneOut
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# XGBoost needs OpenMP on macOS; handle missing-lib gracefully so the notebook still runs.
try:
    from xgboost import XGBClassifier
    XGBOOST_OK = True
    print("XGBoost: OK")
except Exception as e:
    XGBOOST_OK = False
    print(f"XGBoost: NOT available ({type(e).__name__}). Run `brew install libomp` to enable.")

# --- Build the full set of feature matrices ---
# Scale numeric features (keep sparse-friendly so we can hstack with TF-IDF)
scaler = StandardScaler(with_mean=False)
numeric_scaled = sp.csr_matrix(scaler.fit_transform(numeric_df.values))

# 7 TF-IDF feature sets + numeric + (desc + numeric) = 9 sets total
feature_sets = {name: mat for name, (mat, _vec) in tfidf_sets.items()}
feature_sets['numeric']      = numeric_scaled
feature_sets['desc+numeric'] = sp.hstack([tfidf_sets['desc'][0], numeric_scaled]).tocsr()

print(f"\nTotal feature sets: {len(feature_sets)}")
for name, mat in feature_sets.items():
    print(f"  {name:<22s} shape={mat.shape}")

XGBoost: OK

Total feature sets: 9
  desc                   shape=(100, 51)
  username               shape=(100, 100)
  fullname               shape=(100, 173)
  desc_user              shape=(100, 51)
  desc_fullname          shape=(100, 64)
  user_fullname          shape=(100, 268)
  desc_user_fullname     shape=(100, 64)
  numeric                shape=(100, 11)
  desc+numeric           shape=(100, 62)


In [16]:
# --- Algorithm factory + helpers ---

def make_model(algo_name, balanced):
    """Return a fresh classifier. `balanced` toggles class_weight where supported."""
    cw = 'balanced' if balanced else None
    if algo_name == 'LogReg':
        return LogisticRegression(max_iter=2000, class_weight=cw, random_state=42)
    if algo_name == 'DecisionTree':
        return DecisionTreeClassifier(class_weight=cw, random_state=42)
    if algo_name == 'RandomForest':
        # 100 trees instead of 200 — much faster, almost identical accuracy on 100 samples.
        return RandomForestClassifier(n_estimators=100, class_weight=cw, random_state=42, n_jobs=-1)
    if algo_name == 'SVM':
        # kernel='linear' is 5–10x faster than RBF on sparse TF-IDF and works well for text.
        return SVC(kernel='linear', probability=True, class_weight=cw, random_state=42)
    if algo_name == 'AdaBoost':
        # AdaBoost has no class_weight; we apply sample_weight at fit time for balancing.
        return AdaBoostClassifier(random_state=42)
    if algo_name == 'XGBoost':
        return XGBClassifier(eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1)
    raise ValueError(f"Unknown algorithm: {algo_name}")

ALGOS = ['LogReg', 'DecisionTree', 'RandomForest', 'SVM', 'AdaBoost']
if XGBOOST_OK:
    ALGOS.append('XGBoost')

def class_counts(y):
    """Return counts as {0: n0, 1: n1, 2: n2}, filling missing classes with 0."""
    c = pd.Series(y).value_counts().to_dict()
    return {0: int(c.get(0, 0)), 1: int(c.get(1, 0)), 2: int(c.get(2, 0))}

def sample_weights_for_balance(y):
    """Per-row weights so every class contributes equally — used for AdaBoost/XGBoost."""
    y = np.asarray(y)
    classes, counts = np.unique(y, return_counts=True)
    w_per_class = {c: len(y) / (len(classes) * cnt) for c, cnt in zip(classes, counts)}
    return np.array([w_per_class[v] for v in y])


def run_experiment(X, y, algo_name, balanced, training_type, target_column, feature_set_name, n_classes):
    """Run one CV experiment; return one row matching the spec's CSV columns."""
    y = np.asarray(y)

    # Choose the cross-validator
    if training_type == 'K-Fold':
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        K_val = 5
    else:  # LOOCV
        cv = LeaveOneOut()
        K_val = ''

    all_true, all_pred = [], []
    proba_chunks = []  # list of (model.classes_, proba_block, test_indices)

    accuracy = precision = recall = f1 = auc = None
    try:
        for train_idx, test_idx in cv.split(np.zeros(len(y)), y):
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]
            if len(np.unique(y_tr)) < 2:
                continue  # degenerate fold (only happens on tiny 2-class data)

            model = make_model(algo_name, balanced)
            fit_kwargs = {}
            if balanced and algo_name in ('AdaBoost', 'XGBoost'):
                fit_kwargs['sample_weight'] = sample_weights_for_balance(y_tr)
            model.fit(X_tr, y_tr, **fit_kwargs)

            y_pred = model.predict(X_te)
            all_pred.extend(y_pred)
            all_true.extend(y_te)

            if hasattr(model, 'predict_proba'):
                proba_chunks.append((model.classes_, model.predict_proba(X_te), list(test_idx)))

        if not all_pred:
            raise RuntimeError("no predictions made")

        accuracy  = accuracy_score(all_true, all_pred)
        precision = precision_score(all_true, all_pred, average='weighted', zero_division=0)
        recall    = recall_score(all_true, all_pred, average='weighted', zero_division=0)
        f1        = f1_score(all_true, all_pred, average='weighted', zero_division=0)

        # AUC: reassemble a (n_samples × n_global_classes) probability matrix
        if proba_chunks:
            global_classes = sorted(set(y))
            proba_full = np.zeros((len(y), len(global_classes)))
            for classes_seen, block, idxs in proba_chunks:
                col_map = [global_classes.index(c) for c in classes_seen]
                for k, idx in enumerate(idxs):
                    for j, col in enumerate(col_map):
                        proba_full[idx, col] = block[k, j]
            try:
                if len(global_classes) == 2:
                    auc = roc_auc_score(y, proba_full[:, 1])
                else:
                    auc = roc_auc_score(y, proba_full, multi_class='ovr', average='weighted')
            except Exception:
                auc = None
    except Exception:
        pass

    counts = class_counts(y)
    if n_classes == 2:
        nonzero = [v for k, v in counts.items() if k != 2 and v > 0]
    else:
        nonzero = [v for v in counts.values() if v > 0]
    min_size = min(nonzero) if nonzero else 0

    return {
        'iteration':       1,
        'target_column':   target_column,
        '#classes':        n_classes,
        '#class_0':        counts[0],
        '#class_1':        counts[1],
        '#class_2':        counts[2] if n_classes == 3 else 0,
        'min_class_size':  min_size,
        'training_type':   training_type,
        'K':               K_val,
        'algorithm':       algo_name,
        'feature_set':     feature_set_name,
        'Features_count':  X.shape[1],
        'balanced':        balanced,
        'accuracy':        accuracy,
        'precision':       precision,
        'recall':          recall,
        'F1':              f1,
        'AUC':             auc,
    }

print(f"Algorithms to run: {ALGOS}")

Algorithms to run: ['LogReg', 'DecisionTree', 'RandomForest', 'SVM', 'AdaBoost', 'XGBoost']


In [17]:
# --- Main experiment loop ---
# 3 tasks × {3 classes, 2 classes} × N algos × 9 feature sets × {K-Fold, LOOCV} × {balanced, unbalanced}
# Expect ~1,000–1,300 rows. The CSV is saved every 50 rows so progress is never lost.
import time

TASKS = [
    ('target_population',      df['target_population'].values),
    ('locals_vs_diaspora',     df['locals_vs_diaspora'].values),
    ('person_vs_organization', df['person_vs_organization'].values),
]

COL_ORDER = [
    'iteration', 'target_column', '#classes', '#class_0', '#class_1', '#class_2',
    'min_class_size', 'training_type', 'K', 'algorithm', 'feature_set',
    'Features_count', 'balanced', 'accuracy', 'precision', 'recall', 'F1', 'AUC',
]
CSV_PATH = Path.cwd() / 'experiments_results_iteration_1.csv'

results = []
t0 = time.time()

def save_progress():
    pd.DataFrame(results)[COL_ORDER].to_csv(CSV_PATH, index=False)

for target_name, y_full in TASKS:
    for n_classes in [3, 2]:
        # 2-class mode: drop the "unknown" rows (label == 2)
        if n_classes == 2:
            mask = (y_full != 2)
            y = y_full[mask]
        else:
            mask = None
            y = y_full

        if len(np.unique(y)) < 2:
            print(f"  Skipping {target_name} ({n_classes}cls): only one class present")
            continue

        for fset_name, X_full in feature_sets.items():
            X = X_full[mask] if mask is not None else X_full
            for algo_name in ALGOS:
                for training_type in ['K-Fold', 'LOOCV']:
                    for balanced in [True, False]:
                        row = run_experiment(
                            X, y, algo_name, balanced, training_type,
                            target_name, fset_name, n_classes,
                        )
                        results.append(row)
                        if len(results) % 50 == 0:
                            save_progress()
                            print(f"  ... {len(results)} done  ({time.time()-t0:.0f}s elapsed)  "
                                  f"[saved partial CSV]")

# Final save (catches the last <50 rows)
save_progress()
print(f"\nTotal experiments: {len(results)} in {time.time()-t0:.0f}s")
print(f"Saved to: {CSV_PATH.name}")

  ... 50 done  (111s elapsed)  [saved partial CSV]
  ... 100 done  (240s elapsed)  [saved partial CSV]
  ... 150 done  (384s elapsed)  [saved partial CSV]
  ... 200 done  (528s elapsed)  [saved partial CSV]
  ... 250 done  (855s elapsed)  [saved partial CSV]
  ... 300 done  (931s elapsed)  [saved partial CSV]
  ... 350 done  (1214s elapsed)  [saved partial CSV]
  ... 400 done  (1326s elapsed)  [saved partial CSV]
  ... 450 done  (1996s elapsed)  [saved partial CSV]
  ... 500 done  (2080s elapsed)  [saved partial CSV]
  ... 550 done  (2270s elapsed)  [saved partial CSV]
  ... 600 done  (3786s elapsed)  [saved partial CSV]
  ... 650 done  (4510s elapsed)  [saved partial CSV]
  ... 700 done  (4516s elapsed)  [saved partial CSV]
  ... 750 done  (4522s elapsed)  [saved partial CSV]
  ... 800 done  (4527s elapsed)  [saved partial CSV]
  ... 850 done  (4533s elapsed)  [saved partial CSV]
  ... 900 done  (4621s elapsed)  [saved partial CSV]
  ... 950 done  (4711s elapsed)  [saved partial CSV]


In [ ]:
# --- Save results CSV with exactly the columns the PDF (page 13) demands ---
results_df = pd.DataFrame(results)
col_order = [
    'iteration', 'target_column', '#classes', '#class_0', '#class_1', '#class_2',
    'min_class_size', 'training_type', 'K', 'algorithm', 'feature_set',
    'Features_count', 'balanced', 'accuracy', 'precision', 'recall', 'F1', 'AUC',
]
results_df = results_df[col_order]

out = Path.cwd() / 'experiments_results_iteration_1.csv'
results_df.to_csv(out, index=False)
print(f"Saved {len(results_df)} rows to {out.name}")

# --- Show the top-5 model per task by F1 (only models that produced predictions) ---
for tgt in results_df['target_column'].unique():
    print(f"\nTop 5 by F1 for {tgt}:")
    sub = (results_df[results_df['target_column'] == tgt]
           .dropna(subset=['F1'])
           .sort_values('F1', ascending=False))
    cols = ['algorithm', 'feature_set', 'training_type', 'balanced', '#classes',
            'accuracy', 'F1', 'AUC']
    print(sub[cols].head(5).to_string(index=False))